In [1]:
%pip install neo4j


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from neo4j import GraphDatabase

# Configuración de conexión (Asegúrate de que coincida con tu YAML)
uri = "bolt://localhost:7687"
user = "neo4j"
password = "grafo123" 

driver = GraphDatabase.driver(uri, auth=(user, password))

def ejecutar_carga_completa():
    with driver.session() as session:
        # 1. Limpieza total para un escenario limpio
        session.run("MATCH (n) DETACH DELETE n")
        
        # 2. Creación de nodos y rutas (Niveles: Almacén -> Intersección -> Punto Entrega)
        # Se incluyen variaciones de tráfico y capacidad para los desafíos técnicos
        query = """
        CREATE (a:Almacen {id: 'ALM_A', nombre: 'Centro Distribución Norte', x: 0, y: 10})
        CREATE (i1:Interseccion {id: 'INT_1', nombre: 'Cruce Av. Bolívar', x: 5, y: 5})
        CREATE (i2:Interseccion {id: 'INT_2', nombre: 'Redoma Chilemex', x: 10, y: 5})
        CREATE (p:PuntoEntrega {id: 'ENT_B', nombre: 'Supermercado Central', x: 15, y: 0})
        
        // Ruta A: Directa pero con tráfico pesado
        CREATE (a)-[:CONECTA_A {distancia: 12.0, tiempo_estimado: 20, estado_trafico: 0.9, capacidad_max_ton: 30.0}]->(p)
        
        // Ruta B: Vía Intersecciones (Más larga pero fluida)
        CREATE (a)-[:CONECTA_A {distancia: 6.0, tiempo_estimado: 8, estado_trafico: 0.2, capacidad_max_ton: 25.0}]->(i1)
        CREATE (i1)-[:CONECTA_A {distancia: 5.5, tiempo_estimado: 7, estado_trafico: 0.3, capacidad_max_ton: 15.0}]->(i2)
        CREATE (i2)-[:CONECTA_A {distancia: 4.0, tiempo_estimado: 5, estado_trafico: 0.2, capacidad_max_ton: 10.0}]->(p)
        """
        session.run(query)
        print("✅ Escenario logístico cargado con éxito.")

ejecutar_carga_completa()

✅ Escenario logístico cargado con éxito.


In [17]:
def proyectar_grafo():
    with driver.session() as session:
        # Borrar proyección si ya existe para evitar errores
        session.run("CALL gds.graph.drop('grafo_logistica', false)")
        
        # Crear la proyección con las propiedades necesarias para Dijkstra y A*
        query = """
        CALL gds.graph.project(
            'grafo_logistica',
            ['Almacen', 'Interseccion', 'PuntoEntrega'],
            {
            CONECTA_A: {
                orientation: 'NATURAL',
                properties: ['distancia', 'tiempo_estimado', 'estado_trafico', 'capacidad_max_ton']
            }
            },
            {
            nodeProperties: ['x', 'y'] // Necesario para la heurística de A*
            }
        )
        """
        result = session.run(query)
        print("✅ Grafo proyectado en memoria exitosamente.")

proyectar_grafo()

Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description="The query used a deprecated field from a procedure. ('schema' returned by 'gds.graph.drop' is deprecated.)", position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'column': 1, 'offset': 0, 'line': 1}}> for query: "CALL gds.graph.drop('grafo_logistica', false)"


✅ Grafo proyectado en memoria exitosamente.


In [18]:
def forzar_proyeccion():
    with driver.session() as session:
        # Paso A: Eliminar proyecciones viejas
        session.run("CALL gds.graph.drop('grafo_logistica', false)")
        
        # Paso B: Proyección con mapeo detallado de propiedades (Requisito para GDS 2.6.9)
        query = """
        CALL gds.graph.project(
            'grafo_logistica',
            {
                Almacen: { label: 'Almacen', properties: ['x', 'y'] },
                Interseccion: { label: 'Interseccion', properties: ['x', 'y'] },
                PuntoEntrega: { label: 'PuntoEntrega', properties: ['x', 'y'] }
            },
            {
                CONECTA_A: {
                    type: 'CONECTA_A',
                    orientation: 'NATURAL',
                    properties: {
                        distancia: { property: 'distancia' },
                        tiempo_estimado: { property: 'tiempo_estimado' },
                        estado_trafico: { property: 'estado_trafico' }
                    }
                }
            }
        )
        """
        session.run(query)
        print("✅ Grafo proyectado con éxito usando mapeo explícito.")

forzar_proyeccion()

Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description="The query used a deprecated field from a procedure. ('schema' returned by 'gds.graph.drop' is deprecated.)", position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'column': 1, 'offset': 0, 'line': 1}}> for query: "CALL gds.graph.drop('grafo_logistica', false)"


✅ Grafo proyectado con éxito usando mapeo explícito.


In [19]:
#Comparación entre ruta más corta (física) y la más inteligente (tráfico)
def ejecutar_comparativa():
    with driver.session() as session:
        query = """
        MATCH (startNode:Almacen {id: 'ALM_A'}), (endNode:PuntoEntrega {id: 'ENT_B'})
        
        // Algoritmo Dijkstra: Basado en distancia (Criterio Mínimo)
        CALL gds.shortestPath.dijkstra.stream('grafo_logistica', {
            sourceNode: startNode,
            targetNode: endNode,
            relationshipWeightProperty: 'distancia'
        })
        YIELD totalCost AS km
        
        // Algoritmo A*: Basado en tiempo/tráfico (Desafío Técnico #1)
        CALL gds.shortestPath.astar.stream('grafo_logistica', {
            sourceNode: startNode,
            targetNode: endNode,
            latitudeProperty: 'y',
            longitudeProperty: 'x',
            relationshipWeightProperty: 'tiempo_estimado'
        })
        YIELD totalCost AS minutos
        
        RETURN km, minutos
        """
        result = session.run(query)
        res = result.single()
        if res:
            print(f"--- 📊 RESULTADOS DEL OPTIMIZADOR ---")
            print(f"📏 Ruta Dijkstra (Física): {res['km']} km")
            print(f"⏱️ Ruta A* (Inteligente): {res['minutos']} min")

ejecutar_comparativa()

--- 📊 RESULTADOS DEL OPTIMIZADOR ---
📏 Ruta Dijkstra (Física): 12.0 km
⏱️ Ruta A* (Inteligente): 20.0 min


In [ ]:
#Consulta 1: Análisis de Riesgo de Tráfico
query1 = """
MATCH (a)-[r:CONECTA_A]->(b)
WITH r, r.estado_trafico AS trafico 
WHERE trafico > 0.7
RETURN type(r) as Relacion, trafico AS Nivel_Alerta
"""
with driver.session() as session:
    result = session.run(query1)
    print("--- 🚦 Tramos con Tráfico Crítico (> 0.7) ---")
    for record in result:
        print(f"Alerta en tramo {record['Relacion']}: Nivel {record['Nivel_Alerta']}")

--- 🚦 Tramos con Tráfico Crítico (> 0.7) ---
Alerta en tramo CONECTA_A: Nivel 0.9


In [11]:
# Consulta 2: Desglose de Propiedades
query2 = """
UNWIND ['distancia', 'tiempo_estimado', 'estado_trafico'] AS propiedad
MATCH (n:Almacen)-[r:CONECTA_A]->()
RETURN propiedad, r[propiedad] AS Valor
"""
with driver.session() as session:
    result = session.run(query2)
    print("--- 📋 Desglose de Propiedades Dinámicas ---")
    for record in result:
        print(f"Propiedad: {record['propiedad']} | Valor: {record['Valor']}")

--- 📋 Desglose de Propiedades Dinámicas ---
Propiedad: distancia | Valor: 12.0
Propiedad: distancia | Valor: 6.0
Propiedad: tiempo_estimado | Valor: 20
Propiedad: tiempo_estimado | Valor: 8
Propiedad: estado_trafico | Valor: 0.9
Propiedad: estado_trafico | Valor: 0.2


In [12]:
# Consulta 3: Reporte de Capacidad de Carga
query3 = """
MATCH path = (a:Almacen)-[:CONECTA_A*1..3]->(p:PuntoEntrega)
WHERE ALL(r IN relationships(path) WHERE r.capacidad_max_ton >= 15)
RETURN [n in nodes(path) | n.nombre] as Ruta, length(path) as Saltos
"""
with driver.session() as session:
    result = session.run(query3)
    print("--- 🚚 Rutas Disponibles para Carga Pesada (>= 15 ton) ---")
    for record in result:
        print(f"Ruta: {record['Ruta']} | Saltos: {record['Saltos']}")

--- 🚚 Rutas Disponibles para Carga Pesada (>= 15 ton) ---
Ruta: ['Centro Distribución Norte', 'Supermercado Central'] | Saltos: 1


In [13]:
# Consulta 4: Cálculo de Costo de Ruta Personalizado
query4 = """
MATCH (a)-[r:CONECTA_A]->(b)
WITH r, (r.distancia * (1 + r.estado_trafico)) AS costo_calculado
RETURN startNode(r).nombre AS Origen, endNode(r).nombre AS Destino, costo_calculado
"""
with driver.session() as session:
    result = session.run(query4)
    print("--- 💰 Análisis de Costos por Tramo (Distancia x Tráfico) ---")
    for record in result:
        print(f"De {record['Origen']} a {record['Destino']}: ${record['costo_calculado']:.2f}")

--- 💰 Análisis de Costos por Tramo (Distancia x Tráfico) ---
De Centro Distribución Norte a Supermercado Central: $22.80
De Centro Distribución Norte a Cruce Av. Bolívar: $7.20
De Cruce Av. Bolívar a Redoma Chilemex: $7.15
De Redoma Chilemex a Supermercado Central: $4.80


In [14]:
# Consulta 5: Auditoría de la Proyección
query5 = """
CALL gds.graph.list() 
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
"""
with driver.session() as session:
    result = session.run(query5)
    print("--- 🛠️ Estado de la Proyección GDS en Memoria ---")
    for record in result:
        print(f"Grafo: {record['graphName']} | Nodos: {record['nodeCount']} | Relaciones: {record['relationshipCount']}")

--- 🛠️ Estado de la Proyección GDS en Memoria ---
Grafo: grafo_logistica | Nodos: 4 | Relaciones: 4
